In [1]:
import pandas as pd
import json
import os

import requests
from PIL import Image
from io import BytesIO

import math

Image.MAX_IMAGE_PIXELS = None

In [2]:
def download_images(image_urls):
    images = []
    for url in image_urls:
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an error for bad responses
            img = Image.open(BytesIO(response.content))
            images.append(img)
        except Exception as e:
            print(f"Failed to download or process image from {url}: {e}")
    return images

In [3]:
def combine_images_into_grid(images, output_path):
    if not images:
        print("No images to combine.")
        return
    
    # Calculate grid dimensions (rows and columns)
    num_images = len(images)
    grid_columns = math.ceil(math.sqrt(num_images))  # Number of columns
    grid_rows = math.ceil(num_images / grid_columns)  # Number of rows

    # Resize all images to the same size (optional, for uniformity)
    img_width, img_height = images[0].size
    images = [img.resize((img_width, img_height)) for img in images]

    # Create a blank canvas for the grid
    grid_width = grid_columns * img_width
    grid_height = grid_rows * img_height
    combined_image = Image.new("RGB", (grid_width, grid_height))

    # Paste images into the grid
    for idx, img in enumerate(images):
        row = idx // grid_columns
        col = idx % grid_columns
        x_offset = col * img_width
        y_offset = row * img_height
        combined_image.paste(img, (x_offset, y_offset))

    # Save the final image
    combined_image.save(output_path)
    print(f"Combined image saved at {output_path}")

In [4]:
df = pd.read_csv('./data/dimension_tables/dim_artwork.csv')

In [5]:
df.columns

Index(['Unnamed: 0', 'artwork_id', 'title', 'creation_year_start',
       'creation_year_end', 'medium', 'dimension', 'original_image_url',
       'compressed_image_url', 'thumbnail_image_url', 'small_image_url',
       'intro', 'overview', 'style', 'theme', 'main_objects', 'location',
       'background_color', 'artist_id', 'genre_id', 'nudity_id', 'color_id'],
      dtype='object')

In [16]:
# for all subfolders in ./output, open it
# for each json file in the subfolder, open it
exhibitions = {}
out = 'output'
for folder in os.listdir(f'./{out}'):
    if folder == '.DS_Store' or folder.endswith('.zip'):
        continue
    index = 0
    for file in os.listdir(f'./{out}/' + folder):
        if not file.endswith('.json'):
            continue
        output_path = f'./{out}/' + folder + '/' + file.replace('.json', '.jpg')
        if os.path.exists(output_path):
            continue
        with open(f'./{out}/' + folder + '/' + file) as f:
            try:
                data = json.load(f)
            except Exception as e:
                print(f)
                raise e
            art_pieces = data['art_pieces']
            art_urls = []
            for art_piece in art_pieces:
                art_urls.append(df[df['artwork_id'] == art_piece]['compressed_image_url'].values[0])
            images = download_images(art_urls)
            
            combine_images_into_grid(images, output_path)
        # break
    # break

Combined image saved at ./output/I want to see happiness and joy/Exhibition_0.jpg
Combined image saved at ./output/I want to see happiness and joy/Exhibition_1.jpg
Combined image saved at ./output/I want to see happiness and joy/Exhibition_2.jpg
